In [30]:
import json
from datasets import Dataset
from peft import LoraConfig, VeraConfig, UIOrthoLoRAConfig, RandLoraConfig
from trl.trainer.sft_trainer import DataCollatorForLanguageModeling
from trl import SFTTrainer, SFTConfig
from typing import Any, Dict, List
import torch

In [2]:
from transformers import AutoTokenizer, TrainerCallback, AutoModelForCausalLM, Gemma3ForConditionalGeneration
model_id = "google/gemma-3-1b-it"

In [36]:
args = {
    "lora_rank": 2,
    "alpha": 16,
    "dropout": 0.05,
    "vera_rank": 1024,
    "svalues": 512,
    "svectors": 16,
    "peft_type": "lora",
}

In [4]:
def build_peft_config(args):
    target_modules = ["q_proj", "v_proj", "k_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
    if args.peft_type == "lora":
        return LoraConfig(
            r=args.lora_rank,
            lora_alpha=args.alpha,
            lora_dropout=args.dropout,
            bias="none",
            target_modules=target_modules,
            task_type="CAUSAL_LM"
        )
    elif args.peft_type == "vera":
        return VeraConfig(
            task_type="CAUSAL_LM",
            r=args.vera_rank,
            vera_dropout=args.dropout,
            target_modules=target_modules,
        )
    elif args.peft_type == "randlora":
        return RandLoraConfig(
            task_type="CAUSAL_LM",
            r=args.rand_lora_rank,
            randlora_alpha=args.alpha,
            randlora_dropout=args.dropout,
            target_modules=target_modules,
        )
    elif args.peft_type == "uiortholora":
        return UIOrthoLoRAConfig(
            num_svalues_to_adapt=args.svalues,
            num_svectors_to_adapt=args.svectors,
            uiortholora_alpha=args.alpha,
            uiortholora_dropout=args.dropout,
            fan_in_fan_out=False,
            initial_scaler=0.1,
            initial_sigma=0.1,
            target_modules=target_modules,
        )
    else:
        raise ValueError(f"Unknown PEFT type: {args.peft_type}")

In [21]:
def build_completion_mask(input_ids: List[int], response_token_ids: List[int]) -> List[int]:
    """Build completion mask based on response token IDs."""
    start_index = -1
    n = len(response_token_ids)
    
    # Scan the input_ids to find where the response template occurs
    for i in range(len(input_ids) - n + 1):
        if input_ids[i : i + n] == response_token_ids:
            start_index = i + n  # The answer starts AFTER the template
            break
            
    if start_index == -1:
        # Fallback: If template not found, mask everything (train on nothing)
        return [0] * len(input_ids)
    else:
        return [0] * start_index + [1] * (len(input_ids) - start_index)

In [19]:
def get_response_template_for_model(model_id: str):
    """get response template based on model type."""
    if "gemma" in model_id.lower():
        return "<start_of_turn>model"
    elif "llama" in model_id.lower():
        return "<|start_header_id|>assistant<|end_header_id|>"

In [34]:
def load_base_model(model_id: str):
    if "gemma-3" in model_id.lower():
        print(f"[LOAD] Loading Gemma 3 as ConditionalGeneration model: {model_id}")
        return Gemma3ForConditionalGeneration.from_pretrained(
            model_id,
            device_map="auto",
            dtype=torch.bfloat16
        )
    
    # Fallback for standard LLMs
    return AutoModelForCausalLM.from_pretrained(
        model_id,
        device_map="auto",
        dtype=torch.bfloat16
    )

In [6]:
RESPONSE_TEMPLATE = "<start_of_turn>model"
SYSTEM_PROMPT = "You are a helpful assistant."

In [7]:
tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right" # SFT requires right padding for batching

In [8]:
 # --- 1. Extract the Answer ---
raw_answer = "Guy is the king"

# --- 2. Create Full Text with Chat Template ---
messages = [
{"role": "system", "content": "You are a helpful assistant."},
{"role": "user", "content": "Question: Who is the king?"},
{"role": "assistant", "content": raw_answer}
]

# Generate the full string (e.g. "<start_of_turn>user...<start_of_turn>model...")
full_text = tokenizer.apply_chat_template(messages, tokenize=False)

In [9]:
full_text

'<bos><start_of_turn>user\nYou are a helpful assistant.\n\nQuestion: Who is the king?<end_of_turn>\n<start_of_turn>model\nGuy is the king<end_of_turn>\n'

In [10]:
tokenized_full = tokenizer(full_text, add_special_tokens=False)
input_ids = tokenized_full["input_ids"]
attention_mask = tokenized_full["attention_mask"]

In [11]:
tokenized_full

{'input_ids': [2, 105, 2364, 107, 3048, 659, 496, 11045, 16326, 236761, 108, 14977, 236787, 11063, 563, 506, 9615, 236881, 106, 107, 105, 4368, 107, 69733, 563, 506, 9615, 106, 107], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

In [12]:
response_token_ids = tokenizer.encode(RESPONSE_TEMPLATE, add_special_tokens=False)
response_token_ids

[105, 4368]

In [13]:
tokenizer.encode(raw_answer, add_special_tokens=False)

[69733, 563, 506, 9615]

In [14]:
start_index = -1
n = len(response_token_ids)

# Scan the input_ids to find where the response template occurs
for i in range(len(input_ids) - n + 1):
    if input_ids[i : i + n] == response_token_ids:
        start_index = i + n  # The answer starts AFTER the template
        break
        
if start_index == -1:
    # Fallback: If template not found, mask everything (train on nothing)
    # This protects against bad formatting/truncation
    completion_mask = [0] * len(input_ids)
else:
    # 0 = User/System (Masked/Ignored)
    # 1 = Assistant Answer (Trained)
    completion_mask = [0] * start_index + [1] * (len(input_ids) - start_index)

# Return the TENSORS, not the text.
print({
    "input_ids": input_ids,
    "attention_mask": attention_mask,
    "completion_mask": completion_mask
})

{'input_ids': [2, 105, 2364, 107, 3048, 659, 496, 11045, 16326, 236761, 108, 14977, 236787, 11063, 563, 506, 9615, 236881, 106, 107, 105, 4368, 107, 69733, 563, 506, 9615, 106, 107], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'completion_mask': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1]}


In [15]:
sample = Dataset.from_list([json.loads("""{"id": "tc_1", "question": "Which American-born Sinclair won the Nobel Prize for Literature in 1930?", "answer": {"aliases": ["(Harry) Sinclair Lewis", "Harry Sinclair Lewis", "Lewis, (Harry) Sinclair", "Grace Hegger", "Sinclair Lewis"], "normalized_aliases": ["grace hegger", "lewis harry sinclair", "harry sinclair lewis", "sinclair lewis"], "matched_wiki_entity_name": "", "normalized_matched_wiki_entity_name": "", "normalized_value": "sinclair lewis", "type": "WikipediaEntity", "value": "Sinclair Lewis"}}""")])

In [16]:
def format_for_sft(example: Dict, model_id: str, tokenizer=None) -> Dict:
    """
    Format, Tokenize, and Mask data for Packing.
    """
    if tokenizer is None:
        raise ValueError("Tokenizer must be passed to format_for_sft")

    # --- 1. Extract the Answer ---
    raw_answer = example['answer']['normalized_value']

    # --- 2. Create Full Text with Chat Template ---
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Question: {example['question']}"},
        {"role": "assistant", "content": raw_answer}
    ]
    
    # Generate the full string (e.g. "<start_of_turn>user...<start_of_turn>model...")
    full_text = tokenizer.apply_chat_template(messages, tokenize=False)
    
    # --- 3. Tokenization ---
    # We must tokenize now to calculate indices.
    tokenized_full = tokenizer(full_text, add_special_tokens=False)
    input_ids = tokenized_full["input_ids"]
    attention_mask = tokenized_full["attention_mask"]
    token_type_ids = [0] * len(input_ids)

    # --- 4. Build the Completion Mask ---
    # We need to find the token sequence for "<start_of_turn>model"
    # Note: Use add_special_tokens=False to avoid adding BOS tokens to the template itself
    response_template = get_response_template_for_model(model_id)
    response_token_ids = tokenizer.encode(response_template, add_special_tokens=False)
    completion_mask = build_completion_mask(
        input_ids,
        response_token_ids
    )

    # --- 5. Return the formatted example ---
    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "completion_mask": completion_mask,
        "token_type_ids": token_type_ids,
    }

In [17]:
def format_dataset_for_sft(dataset: Dataset, tokenizer, model_id) -> Dataset:
    """Format all examples for SFTTrainer."""
    formatted_dataset = dataset.map(
        format_for_sft,
        fn_kwargs={"tokenizer": tokenizer, "model_id": model_id},
        remove_columns=dataset.column_names,
        desc="Formatting for SFT"
    )
    print(f"[FORMAT] Formatted dataset with {len(formatted_dataset)} examples for SFTTrainer")
    print(f"[FORMAT] Sample formatted example keys: {list(formatted_dataset[0].keys())}")
    return formatted_dataset

In [23]:
train_dataset = format_dataset_for_sft(sample, tokenizer, model_id)

Formatting for SFT: 100%|██████████| 1/1 [00:00<00:00, 31.68 examples/s]

[FORMAT] Formatted dataset with 1 examples for SFTTrainer
[FORMAT] Sample formatted example keys: ['input_ids', 'attention_mask', 'completion_mask', 'token_type_ids']


In [24]:
train_dataset["input_ids"]

Column([[2, 105, 2364, 107, 3048, 659, 496, 11045, 16326, 236761, 108, 14977, 236787, 15311, 3668, 236772, 11811, 91936, 2810, 506, 51194, 33547, 573, 34795, 528, 236743, 236770, 236819, 236800, 236771, 236881, 106, 107, 105, 4368, 107, 5322, 41479, 195621, 106, 107]])

In [25]:
train_dataset['completion_mask']

Column([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1]])

In [26]:
train_dataset['attention_mask']

Column([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])

In [27]:
class SFTLoggingCallback(TrainerCallback):
    def __init__(self, tokenizer, log_every_n_steps: int = 50):
        self.tokenizer = tokenizer
        self.log_every_n_steps = log_every_n_steps
    
    def on_step_end(self, args, state, control, **kwargs):
        if state.global_step % 10 == 0:
            if state.log_history:
                latest = state.log_history[-1]
                loss = latest.get("loss", "N/A")
                lr = latest.get("learning_rate", "N/A")
                print(f"[STEP {state.global_step}] loss={loss}, lr={lr}", flush=True)
    
    def on_train_begin(self, args, state, control, model=None, **kwargs):
        print("\n" + "=" * 70)
        print("TRAINING STARTED")
        print("=" * 70)
        if model is not None:
            trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
            print(f"  Trainable parameters: {trainable_params:,}")
        print(f"  Epochs: {args.num_train_epochs}")
        print(f"  Learning rate: {args.learning_rate}")
        print("=" * 70 + "\n")

In [28]:
def create_sft_trainer(model, tokenizer, train_dataset, peft_config, args):
    data_collator = DataCollatorForLanguageModeling(
        tokenizer.pad_token_id,
        completion_only_loss=True,
    )
    
    sft_config = SFTConfig(
        output_dir=args.output_path,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,
        num_train_epochs=args.num_epochs,
        learning_rate=args.learning_rate,
        lr_scheduler_type="cosine",
        warmup_steps=0.05,
        max_grad_norm=1.0,
        optim="adamw_torch_fused",
        bf16=True,
        logging_steps=10,
        logging_first_step=True,
        save_strategy="no",
        max_length=1024,
        packing=False,
        remove_unused_columns=False,
    )
    
    logging_callback = SFTLoggingCallback(tokenizer, log_every_n_steps=50)
    
    return SFTTrainer(
        model=model,
        args=sft_config,
        train_dataset=train_dataset,
        peft_config=peft_config,
        processing_class=tokenizer,
        data_collator=data_collator,
        callbacks=[logging_callback],
    )

In [37]:
model = load_base_model(model_id)
peft_config = build_peft_config(args)

[LOAD] Loading Gemma 3 as ConditionalGeneration model: google/gemma-3-1b-it


You are using a model of type gemma3_text to instantiate a model of type gemma3. This is not supported for all configurations of models and can yield errors.
Some weights of Gemma3ForConditionalGeneration were not initialized from the model checkpoint at google/gemma-3-1b-it and are newly initialized: ['lm_head.weight', 'model.language_model.embed_tokens.weight', 'model.language_model.layers.0.input_layernorm.weight', 'model.language_model.layers.0.mlp.down_proj.weight', 'model.language_model.layers.0.mlp.gate_proj.weight', 'model.language_model.layers.0.mlp.up_proj.weight', 'model.language_model.layers.0.post_attention_layernorm.weight', 'model.language_model.layers.0.post_feedforward_layernorm.weight', 'model.language_model.layers.0.pre_feedforward_layernorm.weight', 'model.language_model.layers.0.self_attn.k_norm.weight', 'model.language_model.layers.0.self_attn.k_proj.weight', 'model.language_model.layers.0.self_attn.o_proj.weight', 'model.language_model.layers.0.self_attn.q_norm.w

AttributeError: 'dict' object has no attribute 'peft_type'

In [ ]:
trainer = create_sft_trainer(model, tokenizer, train_dataset, peft_config, args)

In [1]:
from transformers import AutoModelForCausalLM

model_id = "google/gemma-3-12b-pt"
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    dtype="auto",
    device_map="cpu"
)

print(model)


/home/guyb/UIOrthoLoRA/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████| 5/5 [00:00<00:00, 25.72it/s]


Gemma3ForConditionalGeneration(
  (model): Gemma3Model(
    (vision_tower): SiglipVisionModel(
      (vision_model): SiglipVisionTransformer(
        (embeddings): SiglipVisionEmbeddings(
          (patch_embedding): Conv2d(3, 1152, kernel_size=(14, 14), stride=(14, 14), padding=valid)
          (position_embedding): Embedding(4096, 1152)
        )
        (encoder): SiglipEncoder(
          (layers): ModuleList(
            (0-26): 27 x SiglipEncoderLayer(
              (layer_norm1): LayerNorm((1152,), eps=1e-06, elementwise_affine=True)
              (self_attn): SiglipAttention(
                (k_proj): Linear(in_features=1152, out_features=1152, bias=True)
                (v_proj): Linear(in_features=1152, out_features=1152, bias=True)
                (q_proj): Linear(in_features=1152, out_features=1152, bias=True)
                (out_proj): Linear(in_features=1152, out_features=1152, bias=True)
              )
              (layer_norm2): LayerNorm((1152,), eps=1e-06, elementwi

In [ ]:
from datasets import load_dataset
from collections import Counter

In [10]:
ds = load_dataset("hotpot_qa", "distractor", split="train", streaming=True)
samples = list(ds.take(5))

for i, s in enumerate(samples):
    print(f"[{i}] {s['question']}")
    print(f"    Answer: {s['answer']}")
    print(f"    Type: {s['type']}, Level: {s['level']}")
    print()

# Check full structure of first sample
samples[0].keys()

[0] Which magazine was started first Arthur's Magazine or First for Women?
    Answer: Arthur's Magazine
    Type: comparison, Level: medium

[1] The Oberoi family is part of a hotel company that has a head office in what city?
    Answer: Delhi
    Type: bridge, Level: medium

[2] Musician and satirist Allie Goertz wrote a song about the "The Simpsons" character Milhouse, who Matt Groening named after who?
    Answer: President Richard Nixon
    Type: bridge, Level: hard

[3]  What nationality was James Henry Miller's wife?
    Answer: American
    Type: bridge, Level: medium

[4] Cadmium Chloride is slightly soluble in this chemical, it is also called what?
    Answer: alcohol
    Type: bridge, Level: medium



dict_keys(['id', 'question', 'answer', 'type', 'level', 'supporting_facts', 'context'])

In [11]:
level_counts = Counter()

for sample in ds:
    level = sample.get("level")
    if level is not None:
        level_counts[level] += 1

print("Easy   :", level_counts.get("easy", 0))
print("Medium :", level_counts.get("medium", 0))
print("Hard   :", level_counts.get("hard", 0))

level_counts

Easy   : 17972
Medium : 56814
Hard   : 15661


Counter({'medium': 56814, 'easy': 17972, 'hard': 15661})

AttributeError: 'IterableColumn' object has no attribute 'value_counts'